## set up session and iam user

In [ ]:
import boto3
session = boto3.Session(profile_name="USER_NAME") #change name to IAM user
bedrock = session.client("bedrock", region_name="REGION") #add region

## set up call to bedrock

In [ ]:
rt = session.client("bedrock-agent-runtime", region_name="REGION") #add region
KB_ID = "KB_ID"  # change to user kb id
HAIKU_ARN = "arn:aws:bedrock:us-east-1:145713875802:inference-profile/us.anthropic.claude-haiku-4-5-20251001-v1:0" # change to model of choice

## Create guardrails

In [ ]:
g = bedrock.create_guardrail(
    name="governed-rag-guardrail",
    description="Grounding + out-of-scope refusal for AI governance RAG",
    contextualGroundingPolicyConfig={
        "filtersConfig": [
            {"type": "GROUNDING", "threshold": 0.75},
            {"type": "RELEVANCE", "threshold": 0.75},
        ]
    },
    topicPolicyConfig={
        "topicsConfig": [
            {"name": "LegalAdvice",
             "definition": "Requests for legal advice or interpretation of law for a specific situation.",
             "type": "DENY"},
            {"name": "MedicalAdvice",
             "definition": "Requests for medical advice, diagnosis, or treatment recommendations.",
             "type": "DENY"},
        ]
    },
    blockedInputMessaging="I can only answer questions grounded in the AI governance sources provided.",
    blockedOutputsMessaging="I can't ground that in the available sources, so I won't answer it.",
)
GUARDRAIL_ID = g["guardrailId"]
print(GUARDRAIL_ID)